In [33]:
import sqlite3
import pandas as pd

In [34]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

In [35]:
test_query = """
WITH user_periods AS (
    SELECT 
        t.uid,
        CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before' ELSE 'after' END as time,
        (julianday(t.first_commit_ts) - julianday('1970-01-01')) * 86400 - d.deadlines as diff_seconds
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
),
user_avgs AS (
    SELECT 
        uid,
        time,
        AVG(diff_seconds / 3600.0) as avg_diff
    FROM user_periods
    GROUP BY uid, time
),
qualified AS (
    SELECT uid
    FROM user_avgs
    GROUP BY uid
    HAVING COUNT(DISTINCT time) = 2
)
SELECT 
    a.time,
    AVG(a.avg_diff) as "AVG(diff)"
FROM qualified q
JOIN user_avgs a ON q.uid = a.uid
GROUP BY a.time
ORDER BY a.time DESC
"""

test_results = pd.io.sql.read_sql(test_query, conn)

In [36]:
control_query = """
WITH user_periods AS (
    SELECT 
        c.uid,
        CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before' ELSE 'after' END as time,
        (julianday(c.first_commit_ts) - julianday('1970-01-01')) * 86400 - d.deadlines as diff_seconds
    FROM control c
    JOIN deadlines d ON c.labname = d.labs
    WHERE c.labname != 'project1'
),
user_avgs AS (
    SELECT 
        uid,
        time,
        AVG(diff_seconds / 3600.0) as avg_diff
    FROM user_periods
    GROUP BY uid, time
),
qualified AS (
    SELECT uid
    FROM user_avgs
    GROUP BY uid
    HAVING COUNT(DISTINCT time) = 2
)
SELECT 
    a.time,
    AVG(a.avg_diff) as "AVG(diff)"
FROM qualified q
JOIN user_avgs a ON q.uid = a.uid
GROUP BY a.time
ORDER BY a.time DESC
"""

control_results = pd.io.sql.read_sql(control_query, conn)

In [37]:
conn.close()

In [38]:
print("test_results:")
print(test_results)
print()
print("control_results:")
print(control_results)

test_results:
     time   AVG(diff)
0  before  -66.679398
1   after -100.178032

control_results:
     time  AVG(diff)
0  before -98.467698
1   after -99.803422
